In [1]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
import plotly.graph_objects as go

# ---------------------------------------------------------------------
# Símbolos de Christoffel del paraboloide hiperbólico z = x^2 - y^2
# ---------------------------------------------------------------------
x, y = sp.symbols('x y', real=True)

def superficie(sx, sy):
    """Vector de posición de la superficie paramétrica.

    Parameters
    ----------
    sx, sy : sympy.Symbol
        Parámetros (coordenadas) de la superficie.

    Returns
    -------
    sympy.Matrix
        Vector de posición r(x, y) = (x, y, x^2 - y^2).
    """
    return sp.Matrix([sx, sy, sx**2 - sy**2])

r = superficie(x, y)
# Derivadas parciales del vector de posición (base tangente)
rx, ry = r.diff(x), r.diff(y)

# Primera forma fundamental (first fundamental form): g_ij = r_i · r_j
g = sp.Matrix([
    [sp.simplify(rx.dot(rx)), sp.simplify(rx.dot(ry))],
    [sp.simplify(ry.dot(rx)), sp.simplify(ry.dot(ry))],
])
g_inv = g.inv()

# Γ^k_ij = (1/2) g^{kl} (∂_i g_jl + ∂_j g_il - ∂_l g_ij)
Gamma = [[[0, 0], [0, 0]], [[0, 0], [0, 0]]]
for k in range(2):
    for i in range(2):
        for j in range(2):
            Gamma[k][i][j] = sp.simplify(sp.Rational(1, 2) * sum(
                g_inv[k, l] * (
                    sp.diff(g[j, l], [x, y][i]) +
                    sp.diff(g[i, l], [x, y][j]) -
                    sp.diff(g[i, j], [x, y][l])
                ) for l in range(2)
            ))

# Lambdify para evaluar Gamma numéricamente en el integrador
gamma_fun = sp.lambdify((x, y), sp.MutableDenseNDimArray(Gamma), 'numpy')

# ---------------------------------------------------------------------
# Resolución numérica de las geodesicas
# ---------------------------------------------------------------------
def geodesica(theta, R=1.5, n=400):
    """Resuelve la geodesica del paraboloide z = x^2 - y^2 que pasa por el origen.

    Parameters
    ----------
    theta : float
        Ángulo inicial en radianes del vector velocidad en el plano (x, y).
        Como g = Id en el origen, (cos θ, sin θ) es unitario allí.
    R : float, optional
        Radio máximo de la coordenada (x, y) antes de cortar la integración.
    n : int, optional
        Número de puntos de evaluación de la curva.

    Returns
    -------
    numpy.ndarray
        Array de forma (n, 3) con los puntos (x, y, z) de la geodesica.
    """
    def ecuacion_geodesica(t, s):
        """Sistema de segundo orden reducido a primero: s = (x, y, vx, vy)."""
        px, py, vx, vy = s
        # Γ numérico en el punto actual
        G = np.asarray(gamma_fun(px, py), dtype=float)
        # Aceleración: a^k = -Γ^k_ij · v^i · v^j
        ax = -(G[0, 0, 0] * vx**2 + 2 * G[0, 0, 1] * vx * vy + G[0, 1, 1] * vy**2)
        ay = -(G[1, 0, 0] * vx**2 + 2 * G[1, 0, 1] * vx * vy + G[1, 1, 1] * vy**2)
        return [vx, vy, ax, ay]

    # Evento: parar al salir del disco de radio R en coordenadas (x, y)
    def salir_disco(t, s):
        return s[0]**2 + s[1]**2 - R**2
    salir_disco.terminal = True
    salir_disco.direction = 1

    # Condición inicial: origen con velocidad unitaria (cos θ, sin θ)
    s0 = [0.0, 0.0, np.cos(theta), np.sin(theta)]
    sol = solve_ivp(ecuacion_geodesica, [0, 50], s0, events=salir_disco,
                    dense_output=True, rtol=1e-9, atol=1e-9)

    t = np.linspace(0, sol.t[-1], n)
    gx, gy = sol.sol(t)[0], sol.sol(t)[1]
    return np.column_stack([gx, gy, gx**2 - gy**2])

# ---------------------------------------------------------------------
# Precalculo de los frames: una geodesica por ángulo
# ---------------------------------------------------------------------
angulos = np.linspace(0, np.pi, 19)   # π basta: las geodesicas no están orientadas

# Malla de la superficie
xm = np.linspace(-1.5, 1.5, 80)
ym = np.linspace(-1.5, 1.5, 80)
X, Y = np.meshgrid(xm, ym)
Z = X**2 - Y**2

superficie_traza = go.Surface(x=X, y=Y, z=Z, colorscale="Viridis",
                              opacity=0.75, showscale=False)

frames = []
for i, th in enumerate(angulos):
    pts = geodesica(th)
    frames.append(go.Frame(
        data=[go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                           mode="lines", line=dict(color="red", width=8))],
        traces=[1],               # solo actualiza la traza de la geodesica
        name=f"frame{i}",
    ))

fig = go.Figure(
    data=[superficie_traza,
          go.Scatter3d(x=[0], y=[0], z=[0], mode="markers",
                       marker=dict(size=5, color="white"))],
    frames=frames,
)

fig.update_layout(
    title="Paraboloide hiperbólico: geodesicas que pasan por el origen",
    sliders=[{
        "currentvalue": {"prefix": "θ = ", "suffix": " rad"},
        "steps": [{
            "label": f"{np.degrees(th):.0f}°",
            "method": "animate",
            "args": [[f"frame{i}"],
                     {"mode": "immediate",
                      "frame": {"duration": 0, "redraw": True}}],
        } for i, th in enumerate(angulos)],
    }],
    scene=dict(aspectmode="data"),
)

fig.show()
# fig.write_html("geodesicas.html", include_plotlyjs="cdn")  ← sigue interactivo

ModuleNotFoundError: No module named 'numpy'